生成新版的knowledgemap的数据

In [19]:
import numpy as np
import pandas as pd
import json
data_dict = np.load('../datasets/datasets.npy', allow_pickle=True).item()
works_df = data_dict['作品表']
people_df = data_dict['人物表']
events_df = data_dict['事件表']
location_df = data_dict['地点表']
poems_df = data_dict['金陵历朝诗歌']
print(events_df.columns.tolist())
print(people_df.columns.tolist())
print(location_df.columns.tolist())
print(works_df.columns.tolist())
print(poems_df.columns.tolist())

data_dict_en = np.load('../datasets/datasets_en.npy', allow_pickle=True).item()
works_df_en = data_dict_en['作品表']
people_df_en = data_dict_en['人物表']
events_df_en = data_dict_en['事件表']
location_df_en = data_dict_en['地点表']
poems_df_en = data_dict_en['金陵历朝诗歌']

['事件id', '事件名称', '朝代', '时期', '年号', '年份', '事件简介', '事件内容', '检索词', '事件类别', '特征类别', '一级地点', '二级地点', '相关人物', '相关作品']
['人物id', '名字', '朝代', '时期', '生年', '卒年', '基本信息', '文学身份', '生平事迹', '文学作品', '文学成就_古人评价', '文学成就_今人评价', '所著作品', '相关作品', '相关人物', '相关地点', '相关事件', '作品数']
['地名id', '地名', '繁体地名', '别名', '地名类别', '景观类型', '地理位置', '朝代', '时期', '经度', '纬度', '相关作品', '相关人物', '相关地点', '相关事件', '地名描述']
['作品id', '作品名', '人物id', '名字', '朝代', '时期', '国家', '文体', '写作时间', '创作背景', '内容简介', '相关作品', '相关人物', '相关地点', '相关事件', '诗歌全文']
['id', 'title', 'author', 'content', 'dynasty']


人物身份

In [20]:
person_type_map_df = pd.read_excel('../bqs_only/related_person_type_freq_merge.xlsx', sheet_name=0)
person_type_merge_map = {}
person_type_merge_freq = {}
person_type_merge_list = []
for row_id, row in person_type_map_df.iterrows():
    person_type_merge_map[row['身份']] = row['合并类别']
    if row['合并类别'] not in person_type_merge_freq.keys():
        person_type_merge_freq[row['合并类别']] = 0
    person_type_merge_freq[row['合并类别']] += int(row['频率'])
person_type_merge_list = list(map(lambda x: x[0], person_type_merge_freq.items()))
print(person_type_merge_freq)

{'none': 172, '文': 254, '艺': 95, '术': 71, '僧': 13, '伶': 6, '曲': 25, '将': 2}


去掉poem中的重复诗词

In [21]:
import re
del_poems_list = []
work_content_list = []
for row_id, row in works_df.iterrows():
    if pd.isna(row['诗歌全文']):
        continue
    work_content_list.append(re.sub('\W*', '', row['诗歌全文']))
for row_id, row in poems_df.iterrows():
    if re.sub('\W*', '', row['content']) in work_content_list:
        del_poems_list.append(row['id'])

生成全部的node和link，和无相关信息的object

In [22]:
dynasty_map = {'东吴': '东吴', '六朝': '东吴', '东晋': '晋', '西晋': '晋', '晋': '晋', '南朝': '南朝', '唐': '唐', '南唐': '南唐', '宋': '宋', '元': '元', '明': '明', '清': '清', '现当代': '现当代', '当代': '现当代'}
all_valid_dynasty = list(dynasty_map.keys())
dynasty_list = ['东吴', '晋', '南朝', '唐', '南唐', '宋', '元', '明', '清', '现当代']
dynasty_list_en = ['Eastern Wu', 'Jin', 'Southern Dynasties', 'Tang', 'Southern Tang', 'Song', 'Yuan', 'Ming', 'Qing', 'Modern']
nodes = []
links = []

In [23]:
event_map = {}
person_map = {}
location_map = {}
work_map = {}
poem_map = {}
# related_person = {} # 能够所及到的人物id
# 事件表
for row_id, row in events_df.iterrows():
    if row['朝代'] not in all_valid_dynasty:
        continue
    row_en = events_df_en[events_df_en['事件id'] == row['事件id']].iloc[0]
    id_str = 'event-' + str(row['事件id'])
    cur_object = {}
    cur_object['id'] = id_str
    cur_object['name'] = row_en['事件名称']
    cur_object['type'] = 'event'
    cur_object['dynasty'] = row_en['朝代']
    cur_object['dynasty_id'] = dynasty_list.index(dynasty_map[row['朝代']])
    cur_object['person'] = []
    cur_object['location'] = []
    cur_object['event'] = []
    cur_object['work'] = []
    cur_object['poem'] = []
    cur_object['info'] = []
    cur_object['detail'] = []
    nodes.append({'id': id_str, 'name': cur_object['name'], 'type': cur_object['type'], 'dynasty_id': cur_object['dynasty_id']})
    # 基本信息
    cur_object['info'].append(['Dynasty', cur_object['dynasty']])
    info_list = ['时期', '年号', '年份', '事件类别']
    info_show = ['Period', 'Reign Title', 'Year', 'Category']
    for i, d in enumerate(info_list):
        if not pd.isna(row[d]):
            if d == '时期':
                cur_object['info'].append([info_show[i], row_en[d].strip().split(';')[0]])
            else:
                cur_object['info'].append([info_show[i], str(row_en[d])])
    # detail 事件简介，事件内容
    detail_list = ['事件简介', '事件内容']
    detail_show = ['Introduction', 'Content']
    for i, d in enumerate(detail_list):
        if not pd.isna(row[d]):
            cur_object['detail'].append([detail_show[i], row_en[d].replace('\n', '')])
    # 相关地点
    if not pd.isna(row['二级地点']):
        relative_location = row['二级地点'].strip().split('、')
        relative_location = list(set(relative_location))
        for location in relative_location:
            selected_location_df = location_df[location_df['地名'] == location]
            if selected_location_df.shape[0] != 0 and (not pd.isna(selected_location_df.iloc[0]['经度'])):
                location_id = 'location-' + str(selected_location_df.iloc[0]['地名id'])
                links.append({'source': location_id, 'target': id_str})
    # 相关人物
    if not pd.isna(row['相关人物']):
        relative_person = row['相关人物'].strip().split('、')
        relative_person = list(set(relative_person))
        for person in relative_person:
            selected_people_df = people_df[people_df['名字'] == person]
            if selected_people_df.shape[0] != 0:
                if selected_people_df.iloc[0]['朝代'] not in all_valid_dynasty:
                    continue
                person_id = 'person-' + str(selected_people_df.iloc[0]['人物id'])
                links.append({'source': id_str, 'target': person_id})
                # related_person[person_id] = 1
    # 相关作品
    if not pd.isna(row['相关作品']):
        relative_work = row['相关作品'].strip().split('、')
        relative_work = list(set(relative_work))
        for work in relative_work:
            selected_work_df = works_df[works_df['作品名'] == work]
            if selected_work_df.shape[0] != 0:
                if selected_work_df.iloc[0]['朝代'] not in all_valid_dynasty:
                    continue
                work_id = 'work-' + str(selected_work_df.iloc[0]['作品id'])
                links.append({'source': id_str, 'target': work_id})
    event_map[id_str] = cur_object
# 人物表
for row_id, row in people_df.iterrows():
    if row['朝代'] not in all_valid_dynasty:
        continue
    row_en = people_df_en[people_df_en['人物id'] == row['人物id']].iloc[0]
    id_str = 'person-' + str(row['人物id'])
    cur_object = {}
    cur_object['id'] = id_str
    cur_object['name'] = row['名字']
    cur_object['type'] = 'person'
    cur_object['dynasty'] = row_en['朝代']
    cur_object['dynasty_id'] = dynasty_list.index(dynasty_map[row['朝代']])
    cur_object['person'] = []
    cur_object['location'] = []
    cur_object['event'] = []
    cur_object['work'] = []
    cur_object['poem'] = []
    cur_object['info'] = []
    cur_object['detail'] = []
    cur_object['person_type'] = 'none' # 没有身份的默认值
    nodes.append({'id': id_str, 'name': cur_object['name'], 'type': cur_object['type'], 'dynasty_id': cur_object['dynasty_id']})
    # 基本信息
    cur_object['info'].append(['Dynasty', cur_object['dynasty']])
    info_list = ['时期', '生年', '卒年']
    info_show = ['Period', 'Birth Year', 'Death Year']
    for i, d in enumerate(info_list):
        if not pd.isna(row[d]):
            if d == '时期':
                cur_object['info'].append([info_show[i], row_en[d].strip().split(';')[0]])
            else:
                cur_object['info'].append([info_show[i], str(row_en[d])])
    # 身份
    if not pd.isna(row['文学身份']):
        person_type = row['文学身份'].strip().split('、')[0]
        if person_type in person_type_merge_map.keys():
            cur_object['person_type'] = person_type_merge_map[person_type]
        else:
            cur_object['person_type'] = 'none'
    # 将身份信息添加进Info
    if cur_object['person_type'] != 'none':
        cur_object['info'].append(['Category', cur_object['person_type']])
    else:
        cur_object['info'].append(['Category', 'None'])
    # detail 基本信息，文学作品
    detail_list = ['基本信息', '文学作品']
    detail_show = ['Introduction', 'Works']
    for i, d in enumerate(detail_list):
        if not pd.isna(row[d]):
            cur_object['detail'].append([detail_show[i], row_en[d].replace('\n', '')])
    # 相关地点
    if not pd.isna(row['相关地点']):
        relative_location = row['相关地点'].strip().split('、')
        relative_location = list(set(relative_location))
        for location in relative_location:
            selected_location_df = location_df[location_df['地名'] == location]
            if selected_location_df.shape[0] != 0 and (not pd.isna(selected_location_df.iloc[0]['经度'])):
                location_id = 'location-' + str(selected_location_df.iloc[0]['地名id'])
                links.append({'source': location_id, 'target': id_str})
    # 相关人物
    if not pd.isna(row['相关人物']):
        relative_person = row['相关人物'].strip().split('、')
        relative_person = list(set(relative_person))
        for person in relative_person:
            selected_people_df = people_df[people_df['名字'] == person]
            if selected_people_df.shape[0] != 0:
                if selected_people_df.iloc[0]['朝代'] not in all_valid_dynasty:
                    continue
                person_id = 'person-' + str(selected_people_df.iloc[0]['人物id'])
                links.append({'source': id_str, 'target': person_id})
                # related_person[person_id] = 1
    # 相关事件
    if not pd.isna(row['相关事件']):
        relative_event = row['相关事件'].strip().split('、')
        relative_event = list(set(relative_event))
        for event in relative_event:
            selected_event_df = events_df[events_df['事件名称'] == event]
            if selected_event_df.shape[0] != 0:
                if selected_event_df.iloc[0]['朝代'] not in all_valid_dynasty:
                    continue
                event_id = 'event-' + str(selected_event_df.iloc[0]['事件id'])
                links.append({'source': id_str, 'target': event_id})
    # 相关work
    selected_work_df = works_df[works_df['人物id'] == row['人物id']] # 所著作品
    write_work = {}
    for i in range(selected_work_df.shape[0]):
        if selected_work_df.iloc[i]['朝代'] not in all_valid_dynasty:
            continue
        work_id = 'work-' + str(selected_work_df.iloc[i]['作品id'])
        links.append({'source': id_str, 'target': work_id})
        write_work[selected_work_df.iloc[i]['作品名']] = 1 # 防止相关作品和所著作品重复
    if not pd.isna(row['相关作品']):
        relative_work = row['相关作品'].strip().split('、')
        relative_work = list(set(relative_work))
        for work in relative_work:
            if work in write_work.keys():
                continue
            selected_work_df = works_df[works_df['作品名'] == work]
            if selected_work_df.shape[0] != 0:
                if selected_work_df.iloc[0]['朝代'] not in all_valid_dynasty:
                    continue
                write_work[work] = 1
                work_id = 'work-' + str(selected_work_df.iloc[0]['作品id'])
                links.append({'source': id_str, 'target': work_id})
    # 相关poem
    selected_poem_df = poems_df[poems_df['author'] == row['名字']]
    for i in range(selected_poem_df.shape[0]):
        if ('《' + selected_poem_df.iloc[i]['title'] + '》') in write_work.keys():
            continue
        if selected_poem_df.iloc[i]['id'] in del_poems_list:
            continue
        poem_id = 'poem-' + str(selected_poem_df.iloc[i]['id'])
        links.append({'source': id_str, 'target': poem_id})
    person_map[id_str] = cur_object
# poem表
for row_id, row in poems_df.iterrows():
    if row['id'] in del_poems_list:
        continue
    row_en = poems_df_en[poems_df_en['id'] == row['id']].iloc[0]
    id_str = 'poem-' + str(row['id'])
    cur_object = {}
    cur_object['id'] = id_str
    cur_object['name'] = row['title']
    cur_object['type'] = 'poem'
    cur_object['dynasty'] = row_en['dynasty']
    cur_object['dynasty_id'] = dynasty_list.index(dynasty_map[row['dynasty']])
    cur_object['person'] = []
    cur_object['location'] = []
    cur_object['event'] = []
    cur_object['work'] = []
    cur_object['poem'] = []
    cur_object['info'] = []
    cur_object['detail'] = []
    nodes.append({'id': id_str, 'name': cur_object['name'], 'type': cur_object['type'], 'dynasty_id': cur_object['dynasty_id']})
    # info
    cur_object['info'].append(['Dynasty', row_en['dynasty']])
    cur_object['info'].append(['Author', row['author']])
    # detail
    cur_object['detail'].append(['诗歌全文', row['content']])
    # 相关人物
    selected_people_df = people_df[people_df['名字'] == row['author']]
    if selected_people_df.shape[0] != 0:
        if selected_people_df.iloc[0]['朝代'] in all_valid_dynasty:
            person_id = 'person-' + str(selected_people_df.iloc[0]['人物id'])
            links.append({'source': id_str, 'target': person_id})
            # related_person[person_id] = 1
    poem_map[id_str] = cur_object
# 地点表
for row_id, row in location_df.iterrows():
    row_en = location_df_en[location_df_en['地名id'] == row['地名id']].iloc[0]
    id_str = 'location-' + str(row['地名id'])
    cur_object = {}
    cur_object['id'] = id_str
    cur_object['name'] = row['地名']
    cur_object['type'] = 'location'
    # 所有的地点朝代都设置为东吴，这样不管选择任何朝代保证地点可以正常显示
    cur_object['dynasty'] = 'Eastern Wu'
    cur_object['dynasty_id'] = 0
    cur_object['person'] = []
    cur_object['location'] = []
    cur_object['event'] = []
    cur_object['work'] = []
    cur_object['poem'] = []
    cur_object['info'] = []
    cur_object['detail'] = []
    nodes.append({'id': id_str, 'name': cur_object['name'], 'type': cur_object['type'], 'dynasty_id': cur_object['dynasty_id']})
    # # detail 别名，地名描述
    # detail_list = ['别名', '地名描述']
    # for d in detail_list:
    #     if not pd.isna(row[d]):
    #         cur_object['detail'].append([d, row[d].replace('\n', '')])
    # # 相关地点
    # if not pd.isna(row['相关地点']):
    #     relative_location = row['相关地点'].strip().split('、')
    #     relative_location = list(set(relative_location))
    #     for location in relative_location:
    #         selected_location_df = location_df[location_df['地名'] == location]
    #         if selected_location_df.shape[0] != 0 and (not pd.isna(selected_location_df.iloc[0]['经度'])):
    #             location_id = 'location-' + str(selected_location_df.iloc[0]['地名id'])
    #             cur_location_object = {'id': location_id, 'name': location};
    #             cur_object['location'].append(cur_location_object)
    # 相关人物
    if not pd.isna(row['相关人物']):
        relative_person = row['相关人物'].strip().split('、')
        relative_person = list(set(relative_person))
        for person in relative_person:
            selected_people_df = people_df[people_df['名字'] == person]
            if selected_people_df.shape[0] != 0:
                if selected_people_df.iloc[0]['朝代'] not in all_valid_dynasty:
                    continue
                person_id = 'person-' + str(selected_people_df.iloc[0]['人物id'])
                links.append({'source': id_str, 'target': person_id})
                # related_person[person_id] = 1
    # 相关事件
    if not pd.isna(row['相关事件']):
        relative_event = row['相关事件'].strip().split('、')
        relative_event = list(set(relative_event))
        for event in relative_event:
            selected_event_df = events_df[events_df['事件名称'] == event]
            if selected_event_df.shape[0] != 0:
                if selected_event_df.iloc[0]['朝代'] not in all_valid_dynasty:
                    continue
                event_id = 'event-' + str(selected_event_df.iloc[0]['事件id'])
                links.append({'source': id_str, 'target': event_id})
    # 地点表的相关work和poem单独计算
    location_map[id_str] = cur_object
# 地点表的相关work和poem计算如下
with open('../json/works_list_v2.json', 'r') as f:
    works_list_v2 = json.load(f)
for work_obj in works_list_v2['works']:
    cur_type = work_obj['作品id'].split('-')[0]
    selected_location_df = location_df[location_df['地名'] == work_obj['地点']]
    location_id = 'location-' + str(selected_location_df.iloc[0]['地名id'])
    links.append({'source': location_id, 'target': work_obj['作品id']})
# work表
for row_id, row in works_df.iterrows():
    if row['朝代'] not in all_valid_dynasty:
        continue
    row_en = works_df_en[works_df_en['作品id'] == row['作品id']].iloc[0]
    id_str = 'work-' + str(row['作品id'])
    cur_object = {}
    cur_object['id'] = id_str
    cur_object['name'] = row['作品名']
    cur_object['type'] = 'work'
    cur_object['dynasty'] = row_en['朝代']
    cur_object['dynasty_id'] = dynasty_list.index(dynasty_map[row['朝代']])
    cur_object['person'] = []
    cur_object['location'] = []
    cur_object['event'] = []
    cur_object['work'] = []
    cur_object['poem'] = []
    cur_object['info'] = []
    cur_object['detail'] = []
    nodes.append({'id': id_str, 'name': cur_object['name'], 'type': cur_object['type'], 'dynasty_id': cur_object['dynasty_id']})
    # 基本信息
    cur_object['info'].append(['Dynasty', cur_object['dynasty']])
    info_list = ['名字', '时期', '文体', '写作时间']
    info_show = ['Author', 'Period', 'Category', 'Year']
    for i, d in enumerate(info_list):
        if not pd.isna(row[d]):
            if d == '时期':
                cur_object['info'].append([info_show[i], row_en[d].strip().split(';')[0]])
            elif d == '名字' or d == '文体':
                cur_object['info'].append([info_show[i], str(row[d])])
            else:
                cur_object['info'].append([info_show[i], str(row_en[d])])
    # detail 名字，朝代，文体，创作背景，内容简介
    detail_list = ['创作背景', '内容简介', '诗歌全文']
    detail_show = ['Background', 'Introduction', 'Content']
    for i, d in enumerate(detail_list):
        if not pd.isna(row[d]):
            if d != '诗歌全文':
                cur_object['detail'].append([detail_show[i], row_en[d].replace('\n', '')])
            else:
                cur_object['detail'].append([detail_show[i], row[d].replace('\n', '')])
    # # 相关地点
    # if not pd.isna(row['相关地点']):
    #     relative_location = row['相关地点'].strip().split('、')
    #     relative_location = list(set(relative_location))
    #     for location in relative_location:
    #         selected_location_df = location_df[location_df['地名'] == location]
    #         if selected_location_df.shape[0] != 0 and (not pd.isna(selected_location_df.iloc[0]['经度'])):
    #             location_id = 'location-' + str(selected_location_df.iloc[0]['地名id'])
    #             cur_location_object = {'id': location_id, 'name': location};
    #             cur_object['location'].append(cur_location_object)
    # 相关人物
    if not pd.isna(row['相关人物']) or not pd.isna(row['名字']):
        relative_person = []
        if not pd.isna(row['相关人物']):
            relative_person += row['相关人物'].strip().split('、')
        if not pd.isna(row['名字']):
            relative_person.append(row['名字'])
        relative_person = list(set(relative_person))
        for person in relative_person:
            selected_people_df = people_df[people_df['名字'] == person]
            if selected_people_df.shape[0] != 0:
                if selected_people_df.iloc[0]['朝代'] not in all_valid_dynasty:
                    continue
                person_id = 'person-' + str(selected_people_df.iloc[0]['人物id'])
                links.append({'source': id_str, 'target': person_id})
                # related_person[person_id] = 1
    # 相关事件
    if not pd.isna(row['相关事件']):
        relative_event = row['相关事件'].strip().split('、')
        relative_event = list(set(relative_event))
        for event in relative_event:
            selected_event_df = events_df[events_df['事件名称'] == event]
            if selected_event_df.shape[0] != 0:
                if selected_event_df.iloc[0]['朝代'] not in all_valid_dynasty:
                    continue
                event_id = 'event-' + str(selected_event_df.iloc[0]['事件id'])
                links.append({'source': id_str, 'target': event_id})
    # 相关作品
    if not pd.isna(row['相关作品']):
        relative_work = row['相关作品'].strip().split('、')
        relative_work = list(set(relative_work))
        for work in relative_work:
            selected_work_df = works_df[works_df['作品名'] == work]
            if selected_work_df.shape[0] != 0:
                if selected_work_df.iloc[0]['朝代'] not in all_valid_dynasty:
                    continue
                work_id = 'work-' + str(selected_work_df.iloc[0]['作品id'])
                links.append({'source': id_str, 'target': work_id})
    work_map[id_str] = cur_object
# 合并
knowledge_map = {'event': event_map, 'person': person_map, 'location': location_map, 'work': work_map, 'poem': poem_map, 'person_type_list': person_type_merge_list}

In [ ]:
# links去重
links_copy = links[:]
links_map = {}
links = []
for link in links_copy:
    link_str = link['source'] + ' ' + link['target']
    if link_str not in links_map.keys():
        links.append(link)
        links_map[link_str] = 1
        link_str = link['target'] + ' ' + link['source']
        links_map[link_str] = 1

根据links为各个object补充相关信息，这样直接保证了是双向的

In [ ]:
for link in links:
    source_id = link['source']
    target_id = link['target']
    source_type = source_id.split('-')[0]
    target_type = target_id.split('-')[0]
    if source_type == 'location':
        # source
        knowledge_map[source_type][source_id][target_type].append({'id': target_id, 'name': knowledge_map[target_type][target_id]['name'], 'type': knowledge_map[target_type][target_id]['type'], 'dynasty_id': knowledge_map[target_type][target_id]['dynasty_id']})
    else:
        # source
        knowledge_map[source_type][source_id][target_type].append({'id': target_id, 'name': knowledge_map[target_type][target_id]['name'], 'type': knowledge_map[target_type][target_id]['type'], 'dynasty_id': knowledge_map[target_type][target_id]['dynasty_id']})
        # target
        knowledge_map[target_type][target_id][source_type].append({'id': source_id, 'name': knowledge_map[source_type][source_id]['name'], 'type': knowledge_map[source_type][source_id]['type'], 'dynasty_id': knowledge_map[source_type][source_id]['dynasty_id']})

为knowledgemap新建一个关建字location_graph，通过该关键字可以直接画出与location相关的所有点，以及他们之间的关系

In [ ]:
location_graph = {}
for location_id, location_obj in knowledge_map['location'].items():
    location_graph[location_id] = {'nodes': [], 'links': []}
# add nodes
for link in links:
    source_id = link['source']
    target_id = link['target']
    source_type = source_id.split('-')[0]
    target_type = target_id.split('-')[0]
    if source_type == 'location':
        location_graph[source_id]['nodes'].append({'id': target_id, 'name': knowledge_map[target_type][target_id]['name'], 'type': knowledge_map[target_type][target_id]['type'], 'dynasty_id': knowledge_map[target_type][target_id]['dynasty_id']})
# add links
for location_id, location_obj in location_graph.items():
    nodes_id_list = list(map(lambda x:x['id'], location_obj['nodes']))
    for link in links:
        source_id = link['source']
        target_id = link['target']
        if (source_id in nodes_id_list) and (target_id in nodes_id_list):
            location_graph[location_id]['links'].append(link)
knowledge_map['location_graph'] = location_graph

In [ ]:

with open('../json/en/knowledge_map_v2.json', 'w') as f:
    json.dump(knowledge_map, f)